In [ ]:
import json
import os
from pathlib import Path
from pprint import pprint
import numpy as np

from datasets import load_from_disk
from transformers import AutoModelForTokenClassification, AutoTokenizer,\
DataCollatorForTokenClassification, TrainingArguments, Trainer, EvalPrediction
import evaluate

from colorama import Fore, Style

In [ ]:
from transformers.models.deberta_v2.tokenization_deberta_v2 import DebertaV2Tokenizer
from transformers.models.deberta_v2.modeling_deberta_v2 import DebertaV2ForTokenClassification
from datasets import DatasetDict

In [ ]:
model_path = "microsoft/deberta-base"

In [ ]:
DATA_DIR = Path(os.getcwd()).parent / "data"
DATA_DIR.mkdir(exist_ok=True)

LABEL_INFO_PATH = DATA_DIR/"label_info.json"
DATASET_PATH = DATA_DIR/"cleaned_ai4privacy_300k_pii"

MODEL_DIR = Path(os.getcwd()).parent / "models"
MODEL_DIR.mkdir(exist_ok=True)
RUN_NAME = "pii_redaction_" + model_path.replace("/", "_") + "_v1"

In [ ]:
dataset: DatasetDict = load_from_disk(DATASET_PATH) # type:ignore

In [ ]:
# label info
with open(LABEL_INFO_PATH) as f:
    label_info: dict = json.load(f)
label2id = label_info["label2id"]
id2label = label_info["id2label"]

In [ ]:
tokenizer: DebertaV2Tokenizer = AutoTokenizer.from_pretrained(model_path)

model: DebertaV2ForTokenClassification = AutoModelForTokenClassification.from_pretrained(
    model_path, num_labels=len(id2label), id2label=id2label, label2id=label2id)

assert tokenizer.is_fast

In [ ]:
def asign_word_ids_to_masks(example: dict[str, list]) -> dict[str, list[list[str]]]:
    enc = tokenizer(
        example["source_text"],
        return_offsets_mapping=True,
        add_special_tokens=True,
    )
    for index, word_id in enumerate(enc.word_ids()):
        for mask in example["source_mask"]:
            

In [ ]:
def get_ner_labels(batch: dict[str, list]) -> dict[str, list[list[str]]]:
    enc = tokenizer(
        batch["source_text"],
        return_offsets_mapping=True,
        add_special_tokens=True,
    )

    token_offsets = enc["offset_mapping"]
    
    batch_ner_labels: list[list[str]] = []
    # loop through the batch to get the privacy masks for every sequence
    for i, row_masks in enumerate(batch["privacy_mask"]):
        row_ner_labels = []
        word_ids = enc.word_ids(batch_index=i)
        
        # loop through the token_offsets of the current sequence
        for offset, token_word_id in zip(token_offsets[i], word_ids):
            # if add_special_tokens is true skip over special tokens (offset with (0,0))
            if offset == (0, 0): 
                row_ner_labels.append("O")
                continue
            # label is "O" unless privacy label is found
            label = "O" 
            # loop through masks to find label
            for privacy_mask in row_masks:
                mask_offset = (privacy_mask["start"], privacy_mask["end"])
                # if statement is switched to check if any character of the token falls within the mask
                if offset[1] > privacy_mask["start"] and offset[0] < privacy_mask["end"]:
                    label = privacy_mask["label"]
            
                    break # break out of for loop when/if label is found
                elif token_word_id == mask_word_id:
                    label = privacy_mask["label"]
                    
                    break
                
            row_ner_labels.append(label)
            
        batch_ner_labels.append(row_ner_labels)
    
    return {"ner_tags": batch_ner_labels}

In [ ]:
dataset = dataset.map(get_ner_labels, batched=True, batch_size=20_000)

In [ ]:
def tokenize_and_align_labels_single(example: dict[str, list]):
    tokenized_input = tokenizer(
        example["source_text"],
        truncation=True,
        add_special_tokens=True
    )
    labels = []
    word_ids = tokenized_input.word_ids()
    previous_word_idx = None
    for i, tag in enumerate(example["ner_tags"]):

        word_idx = word_ids[i]
        
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            # in case there is a label not in label2id it becomes 0
            labels.append(label2id.get(tag, 0))
        else:
            labels.append(-100)
        previous_word_idx = word_idx
            
    tokenized_input["labels"] = labels
    return tokenized_input
            

In [ ]:
dataset = dataset.map(tokenize_and_align_labels_single, batched=False)

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

seqeval = evaluate.load("seqeval")

In [ ]:
from collections import Counter
def compute_metrics(p: EvalPrediction) -> dict[str, float]:
    predictions, label_ids = p.predictions, p.label_ids
    predictions = np.argmax(predictions, axis=-1)
    
    true_preds = [
        [id2label[str(pred)] for pred, tgt_id in zip(preds_row, labels_row) if tgt_id != -100]
        for preds_row, labels_row in zip(predictions, label_ids)
    ]
    
    true_labels =[
        [id2label[str(tgt_id)] for tgt_id in labels_row if tgt_id != -100]
        for labels_row in label_ids
    ]
    
    # TODO: remove
    all_preds = [label for seq in true_preds for label in seq]
    print("Prediction distribution:", Counter(all_preds))
    
    results = seqeval.compute(predictions=true_preds, references=true_labels)
    
    flat = {}
    if results is not None:
        flat.update({
            "precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"],
            "accuracy": results["overall_accuracy"],
        })
        
        for entity, scores in results.items():
            if isinstance(scores, dict): # filter scores for individual labels
                flat[f"{entity}_f1"] = scores["f1"]
                flat[f"{entity}_support"] = scores["number"]
    
    return flat

In [ ]:
small_train = dataset["train"]
small_val = dataset["validation"]

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    learning_rate=1e-5, # low learning_rate
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    warmup_ratio=0.1,
    bf16=False,
    fp16=False,
    max_grad_norm=1.0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer_output = trainer.train()

In [ ]:
import torch
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print(f"NaN in: {name}")
        